# M3L4 E08 — LangGraph básico + Langfuse
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** conectar un grafo simple de LangGraph con Langfuse para observar trazas en tiempo real.

## ¿Cómo funciona la integración?

```
LangGraph graph
    ↓
graph.invoke(..., config={"callbacks": [langfuse_handler]})
    ↓
Langfuse CallbackHandler captura automáticamente:
    - Inputs/outputs de cada nodo
    - Llamadas al LLM (modelo, tokens, latencia)
    - Metadatos de la sesión
    ↓
Langfuse UI → Traces, spans, generations
```

## Jerarquía en Langfuse
```
Trace (toda la ejecución del grafo)
  └── Span (cada nodo)
        └── Generation (cada llamada al LLM)
```

## Paso 1 — Instalación

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalación completa.')

## Paso 2 — Credenciales

> Conseguí tus credenciales en [cloud.langfuse.com](https://cloud.langfuse.com) → Settings → API Keys

Para Europa: `LANGFUSE_BASE_URL = "https://cloud.langfuse.com"`  
Para US: `LANGFUSE_BASE_URL = "https://us.cloud.langfuse.com"`

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key (pk-lf-...): ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key (sk-lf-...): ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')

print('Credenciales configuradas.')

## Paso 3 — LangGraph básico

El grafo más simple posible: un solo nodo que recibe un mensaje y responde.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langfuse.langchain import CallbackHandler

print('Imports OK.')

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('State y LLM listos.')

## TODO — Completar el grafo

Completá los pasos para construir, compilar y ejecutar el grafo.

In [ ]:
def chatbot_node(state: State) -> dict:
    """Nodo que llama al LLM con los mensajes actuales."""
    # TODO: invocar el LLM con state['messages'] y retornar {'messages': [response]}
    pass

print('Nodo definido.')

In [ ]:
# TODO: construir el grafo
# 1. Crear StateGraph con State
# 2. Agregar nodo 'chatbot' con chatbot_node
# 3. set_entry_point('chatbot')
# 4. set_finish_point('chatbot')
# 5. Compilar

graph = None  # reemplazar con la compilación
print('Grafo compilado.')

## Paso 4 — Ejecutar con Langfuse

El truco es pasar el `CallbackHandler` en el `config`.

In [ ]:
langfuse_handler = CallbackHandler()

# TODO: invocar el grafo con:
# - input: {'messages': [HumanMessage(content='Explicame qué es tracing en IA en 2 oraciones')]}
# - config: {'callbacks': [langfuse_handler]}
result = None  # reemplazar

if result:
    print('Respuesta:', result['messages'][-1].content)

## Visualizar el grafo

In [ ]:
if graph:
    print(graph.get_graph().draw_mermaid())

## ¿Qué ver en Langfuse?

Después de ejecutar la celda anterior:
1. Abrí [cloud.langfuse.com](https://cloud.langfuse.com)
2. Andá a **Traces**
3. Buscá la trace recién creada
4. Revisá:
   - **Input:** el mensaje enviado
   - **Output:** la respuesta del LLM
   - **Modelo:** `gpt-4o-mini`
   - **Tokens:** input + output tokens
   - **Duración:** latencia total
   - **Spans:** cada paso del grafo

In [ ]:
assert graph is not None, 'El grafo no fue compilado'
assert result is not None, 'No se obtuvo resultado'
assert len(result['messages']) > 0, 'No hay mensajes en el resultado'
print('Checks E08 OK ✅')